<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Camera-Less Eye Tracking — Benchmark (Akida 2)

Loads the converted `.fbz` eyetracking model, evaluates it on Akida (software backend if no hardware is attached), and — if a physical Akida 2 FPGA board is available — benchmarks latency.


## Setup


In [ ]:
import os
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/processed'
MODEL_FBZ = './pretrained_models/tinytemporalcnn_eyetracking_i8_w8_a8.fbz'
WINDOW = 64


## Load the Akida model


In [ ]:
ak_model = akida.Model(MODEL_FBZ)
ak_model.summary()


## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the software backend is used instead (evaluation still works; latency benchmarking below is skipped).


In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
print('Akida hardware found:', device) if device is not None else print('No hardware found — using software backend')


## Evaluation

Akida's buffered/FIFO layers (`BufferTempConv`) only maintain correct state for **batch_size=1** — sharing one FIFO across a batch of independent windows silently produces garbage. Each window below gets a fresh model instance and is fed one timestep at a time, which is why evaluation is subsampled rather than run over the full test set.


In [ ]:
from eyetracking_data import get_data, to_signed_int8
from eyetracking_eval import angular_error_deg, evaluate_akida_model

scaler = np.load('./pretrained_models/tinytemporalcnn_eyetracking.h5.yscaler.npz')
y_mean, y_std = scaler['mean'], scaler['std']

d = np.load(os.path.join(DATA_PATH, 'test_seq.npz'))
X_test, y_test = to_signed_int8(d['X']), d['y'].astype(np.float32)

rng = np.random.default_rng(0)
n_eval = 300
eval_idx = rng.choice(len(X_test), size=n_eval, replace=False)
X_eval, y_eval = X_test[eval_idx], y_test[eval_idx]

y_pred, ak_model = evaluate_akida_model(ak_model, X_eval)
y_pred = y_pred * y_std + y_mean
overall_err = angular_error_deg(y_eval, y_pred)
last_err = angular_error_deg(y_eval[:, -1], y_pred[:, -1])
print(f'Akida overall mean angular error (n={n_eval}): {overall_err:.3f} deg')
print(f'Akida last-timestep mean angular error: {last_err:.3f} deg')


## Activation Sparsity


In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from eyetracking_data import get_samples

calib_samples = get_samples(DATA_PATH, num_samples=1024)
sparsity_dict = compute_sparsity(ak_model, samples=calib_samples)
pretty_print_sparsity(sparsity_dict)


## Hardware Benchmark

Requires a physical Akida 2 FPGA platform with a connected board. The model is streaming/bufferized, so it must be timed one frame at a time (batch_size=1) rather than with the batched `full_model_benchmark` helper used by the classification examples in this zoo — see `eyetracking_benchmark.py` for the full timing loop and its TODO about getting cycle-accurate (not just wall-clock) latency here.


In [ ]:
import time

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal)
    samples = get_samples(DATA_PATH, num_samples=10 * WINDOW).reshape(10, WINDOW, 1, 1, -1)
    layers = ak_model.layers
    per_window_s = []
    for i in range(len(samples)):
        ak_model = type(ak_model)(layers)  # fresh FIFO state per window
        t0 = time.perf_counter()
        for t in range(WINDOW):
            ak_model.predict(samples[i, t:t + 1])
        per_window_s.append(time.perf_counter() - t0)
    mean_window_ms = float(np.mean(per_window_s)) * 1000
    print(f'Measured @ {AKIDA_CLOCKS_HZ["AKIDA2_FPGA"]/1e6:.0f} MHz FPGA: '
          f'{mean_window_ms:.3f} ms/window ({mean_window_ms * 1000 / WINDOW:.1f} us/frame)')
else:
    print('Hardware not available — skipping benchmark')


## Summary


In [ ]:
print(f'Akida overall mean angular error: {overall_err:.3f} deg')
print(f'Akida last-timestep mean angular error: {last_err:.3f} deg')
print(f'Mean activation sparsity: {np.mean(list(sparsity_dict.values())) * 100:.2f}%')
